In [1]:
import pandas as pd
df = pd.read_csv("report_cdn.csv")

In [2]:
def remap_df(df):
    df = df.copy()

    # 1. Rebalance strategy mapping
    def map_rebalance_strategy(x):
        if x == "marginal-hits-new":
            return "marginal-hits-tuned"
        elif x == "marginal-hits-old":
            return "marginal-hits"
        else:
            return x

    df["rebalance_strategy"] = df["rebalanceStrategy"].apply(map_rebalance_strategy)
    df = df[(df['maxDecayInterval'].isna()) | (df['maxDecayInterval'] == 50_000)]
    df = df[(df['countColdTailHitsOnly'].isna()) | (df['countColdTailHitsOnly'] == True)]
    df = df[(df['mhMovingAverageParam'].isna()) | (df['mhMovingAverageParam'] == 0.3)]

    #Append maxDecayInterval if not null
    def append_max_decay(row):
        val = row.get("maxDecayInterval", None)
        base = row["rebalance_strategy"]
        if pd.notnull(val):
            try:
                val = int(val)
                suffix = f"-{val//1000}k"
            except Exception:
                suffix = f"-{val}"
            return f"{base}{suffix}"
        return base

    #df["rebalance_strategy"] = df.apply(append_max_decay, axis=1)

    # 2. Allocator mapping
    def map_allocator(x):
        if x == "TINYLFUTail":
            return "TINYLFU"
        elif x == "SIMPLE2Q":
            return "LRU"
        else:
            return x

    df["allocator"] = df["allocator"].apply(map_allocator)

    # 3. Tag mapping
    def map_tag(row):
        if row["rebalanceStrategy"] in ["marginal-hits-new", "marginal-hits-old"] and row["allocator"] == "LRU2Q":
            val = row.get("countColdTailHitsOnly", False)
            if pd.notnull(val) and bool(val):
                return "cold"
            else:
                return "warm-cold"
        else:
            return None

    df["tag"] = df.apply(map_tag, axis=1)

    # 4. Add suffix to rebalance_strategy based on threshold fields
    def add_threshold_suffix(row):
        suffix = ""
        ai = row.get("thresholdAI", False)
        ad = row.get("thresholdAD", False)
        mi = row.get("thresholdMI", False)
        md = row.get("thresholdMD", False)
        if pd.notnull(ai) and pd.notnull(ad) and pd.notnull(mi) and pd.notnull(md):
            if bool(ai) and bool(ad):
                suffix = "-aiad"
            elif bool(mi) and bool(md):
                suffix = "-mimd"
            elif bool(ai) and bool(md):
                suffix = "-aimd"
        return str(row["rebalance_strategy"]) + suffix if suffix else row["rebalance_strategy"]
    #df["rebalance_strategy"] = df.apply(add_threshold_suffix, axis=1)

    # 5. Rename columns
    rename_dict = {
        "_missRatio": "miss_ratio",
        "_rebalancerNumRebalancedSlabs": "n_rebalanced_slabs",
        "wakeUpRebalancerEveryXReqs": "monitor_interval",
        "_allocFailures": "n_alloc_failures"
    }
    df = df.rename(columns=rename_dict)

    # 6. Select columns to keep
    keep_cols = [
        "trace_name", "number_of_requests", "wsr", "slab_size", "slab_cnt",
        "rebalance_strategy", "allocator", "tag", "throughput", "rebalanced_slabs",
        "miss_ratio", "n_rebalanced_slabs", "monitor_interval", "n_alloc_failures", "uuid", 
    ]
    keep_cols = [col for col in keep_cols if col in df.columns]
    return df[keep_cols]

In [3]:
import numpy as np

def add_miss_ratio_reduction_from_disabled(df):
    df = df.copy()
    df = df.sort_values(["trace_name", "wsr", "allocator"])
    reduction_series = pd.Series(np.nan, index=df.index)

    group_cols = ["trace_name", "wsr", "allocator"]
    for _, group in df.groupby(group_cols):
        baseline = group[group["rebalance_strategy"] == "disabled"]
        if baseline.empty:
            baseline_miss = np.nan
        else:
            baseline_miss = baseline.iloc[0]["miss_ratio"]
        for idx, row in group.iterrows():
            reduction = baseline_miss - row["miss_ratio"] if pd.notnull(baseline_miss) and pd.notnull(row["miss_ratio"]) else np.nan
            reduction_series.at[idx] = reduction

    df["miss_ratio_reduction_from_disabled"] = reduction_series
    return df

In [4]:
def add_miss_ratio_reduction_from_lru_disabled(df):
    """
    For each group of (trace_name, wsr), use the row with allocator=='LRU' and rebalance_strategy=='disabled' as baseline.
    For each row in the group, compute miss_ratio_reduction_from_lru_disabled = baseline_miss_ratio - row['miss_ratio'].
    """
    df = df.copy()
    reduction_series = pd.Series(np.nan, index=df.index)
    group_cols = ["trace_name", "wsr"]

    # Build a lookup for baseline miss_ratio
    baseline_lookup = (
        df[(df["allocator"] == "LRU") & (df["rebalance_strategy"] == "disabled")]
        .set_index(group_cols)["miss_ratio"]
        .to_dict()
    )

    for idx, row in df.iterrows():
        key = (row["trace_name"], row["wsr"])
        baseline_miss = baseline_lookup.get(key, np.nan)
        if pd.notnull(baseline_miss) and pd.notnull(row["miss_ratio"]):
            reduction_series.at[idx] = baseline_miss - row["miss_ratio"]
        else:
            reduction_series.at[idx] = np.nan

    df["miss_ratio_reduction_from_lru_disabled"] = reduction_series
    return df

In [5]:
def add_tuned_improvement(df):
    df = df.copy()
    mask = df["rebalance_strategy"].str.startswith("marginal-hits-tuned")

    lookup = df[df["rebalance_strategy"] == "marginal-hits"].set_index(
        ["trace_name", "wsr", "allocator", "tag", "monitor_interval"]
    )["miss_ratio"]

    improvements = pd.Series(np.nan, index=df.index)
    for idx, row in df[mask].iterrows():
        key = (row["trace_name"], row["wsr"], row["allocator"], row["tag"], row["monitor_interval"])
        base = lookup.get(key, np.nan)
        if isinstance(base, pd.Series):
            base = base.iloc[0]
        if pd.notnull(base) and pd.notnull(row["miss_ratio"]):
            improvements.at[idx] = base - row["miss_ratio"]
        else:
            improvements.at[idx] = np.nan

    df["tuned_improvement"] = improvements
    return df

In [6]:
remapped_df = remap_df(df)
remapped_df["trace_name"] = remapped_df["trace_name"].replace({
    "meta_202210_kv_traces_all_sort": "meta_202210_kv",
    "meta_202401_kv_traces_all_sort": "meta_202401_kv"
})
remapped_df = add_miss_ratio_reduction_from_disabled(remapped_df)
remapped_df = add_miss_ratio_reduction_from_lru_disabled(remapped_df)
remapped_df = add_tuned_improvement(remapped_df)

In [7]:
remapped_df['tag'].unique()

array([None, 'cold', 'warm-cold'], dtype=object)

In [8]:
remapped_df = remapped_df[remapped_df['tag'] != 'warm-cold']

In [13]:
remapped_df.to_csv("report_cdn_digest.csv", index=False)

In [15]:
remapped_df[remapped_df['trace_name'] == 'tencent_photo2'].sort_values(
    by = ['wsr', 'miss_ratio']
)[['wsr', 'allocator', 'rebalance_strategy', 'miss_ratio', 'n_rebalanced_slabs']]

,wsr,allocator,rebalance_strategy,miss_ratio,n_rebalanced_slabs
126,0.01,LRU2Q,marginal-hits-tuned,0.412175,7297
15,0.01,TINYLFU,marginal-hits-tuned,0.414087,6244
40,0.01,LRU2Q,hits,0.422619,1660
151,0.01,TINYLFU,hits,0.427221,1645
12,0.01,LRU2Q,disabled,0.430022,0
66,0.01,LRU2Q,free-mem,0.430022,0
19,0.01,TINYLFU,disabled,0.434492,0
67,0.01,TINYLFU,free-mem,0.434492,0
147,0.01,TINYLFU,marginal-hits,0.438969,54455
140,0.01,LRU,marginal-hits-tuned,0.440944,6668


In [ ]:
remapped_df[remapped_df["rebalance_strategy"] == "lama"]

,trace_name,number_of_requests,wsr,slab_size,slab_cnt,rebalance_strategy,allocator,tag,throughput,rebalanced_slabs,miss_ratio,n_rebalanced_slabs,monitor_interval,n_alloc_failures,uuid,miss_ratio_reduction_from_disabled,miss_ratio_reduction_from_lru_disabled,tuned_improvement
1261,meta_202210_kv,1482478166,0.005,4,110,lama,LRU,None,11140.371532,1467.0,0.079694,1274,1000000,279824,meta_202210_kv-b2a60eebe41cb747eedc2e6f9bd507e5,0.043899,0.043899,NaN
1270,meta_202210_kv,1482478166,0.010,4,220,lama,LRU,None,15565.626395,441.0,0.073834,53,1000000,0,meta_202210_kv-9559b6e1093f32d5294b697078a0f776,0.026117,0.026117,NaN
1252,meta_202210_kv,1482478166,0.020,4,439,lama,LRU,None,15842.198345,922.0,0.070880,27,1000000,0,meta_202210_kv-33fdb8d282fd324dba8f7a076de16707,0.010508,0.010508,NaN
1266,meta_202210_kv,1482478166,0.050,4,1097,lama,LRU,None,15781.968855,NaN,0.062809,0,1000000,0,meta_202210_kv-bc733591ec01876e836468df175144be,0.000000,0.000000,NaN
1264,meta_202210_kv,1482478166,0.100,4,2193,lama,LRU,None,16233.237400,NaN,0.053112,0,1000000,0,meta_202210_kv-25656eccfdac183fee9c022ccb13ddbc,0.000000,0.000000,NaN
1249,meta_202401_kv,1191241025,0.005,4,59,lama,LRU,None,9058.343302,1235.0,0.131834,1180,1000000,2722378,meta_202401_kv-458fbba9de7431ef3379090ddc02ec0c,0.056963,0.056963,NaN
1253,meta_202401_kv,1191241025,0.010,4,117,lama,LRU,None,9025.825738,1256.0,0.111360,1150,1000000,346034,meta_202401_kv-6e6e9a3be32035559f707be770abcad9,0.042329,0.042329,NaN
1268,meta_202401_kv,1191241025,0.020,4,234,lama,LRU,None,12301.183842,356.0,0.099722,90,1000000,0,meta_202401_kv-3bf733ef2ef5b4994426f7ed715209ba,0.025883,0.025883,NaN
1248,meta_202401_kv,1191241025,0.050,4,583,lama,LRU,None,12401.985815,88.0,0.094761,3,1000000,0,meta_202401_kv-d09bf82e4ce21c1e50e36561b78866de,0.001354,0.001354,NaN
1256,meta_202401_kv,1191241025,0.100,4,1166,lama,LRU,None,12628.917009,NaN,0.077183,0,1000000,0,meta_202401_kv-8ae24a94d061f6750be588c34307946b,0.000000,0.000000,NaN


In [35]:
remapped_df["trace_name"] = remapped_df["trace_name"].replace({
    "meta_202210_kv_traces_all_sort": "meta_202210_kv",
    "meta_202401_kv_traces_all_sort": "meta_202401_kv"
})

In [14]:
remapped_df.to_csv("report_brief_digest.csv", index=False)

In [11]:
remapped_df['trace_name'].unique()

array(['meta_reag', 'meta_rnha', 'meta_rprn', 'wiki_2016u', 'wiki_2019t',
       'wiki_2019u'], dtype=object)

In [17]:
remapped_df[
    (remapped_df["trace_name"] == "wiki_2019t")  
    & (remapped_df["wsr"] == 0.01)
    & (remapped_df['monitor_interval'] == 50_000)
].sort_values(
    by=["trace_name", "wsr", "miss_ratio", 'throughput', "n_rebalanced_slabs"]
)[['rebalance_strategy', 'allocator', 'tag', 'miss_ratio', 'monitor_interval', 'throughput','n_rebalanced_slabs']]

,rebalance_strategy,allocator,tag,miss_ratio,monitor_interval,throughput,n_rebalanced_slabs
73,marginal-hits,TINYLFU,None,0.501482,50000,384429.377579,4148
40,hits,TINYLFU,None,0.507091,50000,159969.986197,459
14,marginal-hits-tuned,TINYLFU,None,0.508030,50000,161162.364474,641
79,disabled,TINYLFU,None,0.519203,50000,339186.075891,0
37,free-mem,TINYLFU,None,0.520080,50000,331615.170788,10
113,tail-age,TINYLFU,None,0.522300,50000,349045.774464,4147
0,marginal-hits-tuned,LRU2Q,cold,0.525742,50000,344046.474889,744
82,marginal-hits,LRU2Q,cold,0.526212,50000,336416.822054,4148
36,hits,LRU2Q,None,0.526573,50000,390217.725808,463
28,disabled,LRU2Q,None,0.536363,50000,438629.766354,0
